In [1]:
import pandas as pd
import numpy as np

In [2]:
libnames = ["CC", "ATF2", "CTCF", "FOXA1", "LEF1", "SCRT1", "TCF7L2", "16P12_1"]
types = ["peaks_vs_notpeaks", "active_vs_inactive", "responsive_vs_nonresponsive", "induced_vs_repressed", "repressed_vs_induced"]
mea_df = []



for i,lib in enumerate(libnames):
    for rtype in types:
        if lib=="CC" and rtype in ["responsive_vs_nonresponsive", "induced_vs_repressed", "repressed_vs_induced"]:
            continue
        elif lib!="CC" and rtype=="active_vs_inactive":
            continue
        ame_file = f"/data7/deepro/starrseq/papers/results/3_motif_enrichment_fragment_category/data/{lib}/homer/{rtype}/knownResults.txt"
        ame_df = pd.read_csv(ame_file, sep="\t", skipfooter=4, usecols=["Motif Name", "q-value (Benjamini)"], engine="python")
        ame_df = ame_df.rename(columns=dict(zip(["Motif Name", "q-value (Benjamini)"], ["motif", "qvalue"])))
        ame_df = ame_df.loc[ame_df["qvalue"]<0.05].sort_values("qvalue")
        ame_df["rank"] = np.arange(1, len(ame_df)+1)
        ame_df["lib"] = lib
        ame_df["comparison"] = rtype
        mea_df.append(ame_df)


In [3]:
mea_df = pd.concat(mea_df)

In [4]:
motif_df = mea_df.pivot(index="rank", columns=["lib", "comparison"], values="motif")
qvalue_df = mea_df.pivot(index="rank", columns=["lib", "comparison"], values="qvalue")

In [6]:
mea_df = pd.concat({"motif": motif_df, "qvalue": qvalue_df}, axis=1).reorder_levels([1, 2, 0], axis=1)
cols = mea_df.columns
lib = pd.CategoricalIndex(cols.get_level_values(0), categories=libnames,  ordered=True)
comp = pd.CategoricalIndex(cols.get_level_values(1), categories=types, ordered=True)
metric = pd.CategoricalIndex(cols.get_level_values(2), categories=["motif", "qvalue"], ordered=True)
mea_df.columns = pd.MultiIndex.from_arrays([lib, comp, metric])

In [12]:
mea_df = mea_df.sort_index(axis=1)

In [13]:
savefile = "/data7/deepro/starrseq/papers/results/3_motif_enrichment_fragment_category/data/tables/supplementary_data2.xlsx"


mea_df.to_excel(savefile)